## **RNN Activations**

### 1. Is there already an activation function inside SimpleRNN?

**Yes, absolutely.** A standard RNN cell has `tanh` baked in **at every single timestep**. The complete mathematical formula executed internally at each step is:

$$h_t = \tanh(W_{ih} \cdot x_t + b_{ih} + W_{hh} \cdot h_{t-1} + b_{hh})$$

In PyTorch, you can verify this directly by looking at the `nn.RNN` signature:
```python
torch.nn.RNN(input_size, hidden_size, nonlinearity='tanh', ...)
#                                      ^^^^^^^^^^^^^^^^^^^^^^^^^
#                                      This is the built-in activation. 
#                                      You can change it to 'relu', but tanh is default
```
This means **you must never add another activation function after each timestep output**. The activation is already happening inside the black box. Adding another one would double-compress the values and destroy the learned representations.

---

### 2. Why `tanh` specifically, and NOT `ReLU`?

This is where your assumption needs a correction. You said:

> *"ReLU variants are used for eliminating vanishing/exploding gradient problems"*

This is true for **feedforward networks (ANNs, CNNs)**, but it is the **wrong thing to use in the recurrent connection** of an RNN. Here is why:

#### The Core Reason: Weight Matrices are Multiplied Repeatedly Through Time

In Backpropagation Through Time (BPTT), the **same weight matrix** $W_{hh}$ is applied at every single timestep. If you have 100 timesteps, the gradient passes through $W_{hh}$ **100 times**.

| Scenario | What happens |
|:---|:---|
| $\|W_{hh}\| < 1$ | Gradient shrinks exponentially → **Vanishing Gradient** |
| $\|W_{hh}\| > 1$ | Gradient grows exponentially → **Exploding Gradient** |

Now consider the two activation functions:
*   **`tanh`**: Squashes output to `[-1, 1]`. This **bounds** the gradients, preventing the most extreme explosions. Vanishing is still a risk for very long sequences though.
*   **`ReLU` in the recurrent path**: If any hidden state value is > 0, the gradient passes through **unchanged** (derivative = 1). Since $W_{hh}$ is multiplied 100 times, even a weight of `1.01` causes the gradient to grow as $1.01^{100} ≈ 2.7$, which **explodes catastrophically**.

> This is why trying `nonlinearity='relu'` in `nn.RNN` almost always leads to exploding gradients and `NaN` losses.

#### The Real Solutions to Vanishing/Exploding Gradients in RNNs:

| Problem | Real Solution |
|:---|:---|
| **Exploding Gradients** | **Gradient Clipping** (`torch.nn.utils.clip_grad_norm_`) |
| **Vanishing Gradients** | **LSTM / GRU** (Gating mechanisms, not just activation functions) |

LSTM and GRU solve the vanishing gradient problem not by using ReLU, but by introducing **learnable gates** (sigmoid-activated) that allow the network to decide how much of the past to remember or forget. The gradient highway through the **cell state** in LSTM remains almost unchanged across thousands of timesteps.

---

### 3. What about the Encoder-Decoder Architecture?

In an Encoder-Decoder (Seq2Seq) model:

```
INPUT → [Encoder RNN] → Context Vector → [Decoder RNN] → OUTPUT
```

*   **Inside the Encoder:** `tanh` is already applied at every timestep. No external activation needed.
*   **The Context Vector:** This is just the raw final hidden state `h_T`. No activation is applied to it.
*   **Inside the Decoder:** `tanh` is applied at every timestep internally.
*   **The Output Linear Layer:** No activation function! The raw logits (scores) are passed directly to `CrossEntropyLoss`, which applies `log_softmax` internally.

```python
# Your current setup is CORRECT:
output = self.linear(final.squeeze(0)) # Raw logits, no activation
loss = CrossEntropyLoss()(output, target) # CE internally applies softmax
```

---

### Summary

| Question | Answer |
|:---|:---|
| Is there an activation in SimpleRNN? | ✅ Yes, `tanh` is built in at every timestep |
| Should you add activation after each timestep output? | ❌ No, it is already there |
| Does ReLU fix vanishing gradients in RNNs? | ❌ No, ReLU causes **exploding** gradients in recurrent paths |
| What actually fixes vanishing gradients in RNNs? | ✅ LSTM / GRU gating mechanisms |
| What fixes exploding gradients in RNNs? | ✅ Gradient Clipping |
| Should you add activation after the final Linear layer? | ❌ No, `CrossEntropyLoss` handles softmax internally |

<div></div>

---

<div></div>

## **Gradient of loss wrt Embeddings (Lookup Table)**

### Setup: Your full pipeline

$$\text{index } i \xrightarrow{\text{lookup}} \mathbf{e}_i \xrightarrow{\text{RNN}} \mathbf{h} \xrightarrow{\text{Linear}} \mathbf{o} \xrightarrow{\text{CrossEntropy}} L$$

Let $E$ be the full embedding matrix of shape `[vocab_size, embed_dim]`. The embedding for word $i$ is simply **row $i$** of $E$:

$$\mathbf{e}_i = E[i] \quad \text{(a vector of 50 numbers)}$$

---

### The Gradient: Chain Rule

By the chain rule, the gradient of the loss $L$ with respect to the embedding vector $\mathbf{e}_i$ is:

$$\frac{\partial L}{\partial \mathbf{e}_i} = \frac{\partial L}{\partial \mathbf{h}} \cdot \frac{\partial \mathbf{h}}{\partial \mathbf{e}_i}$$

Breaking this down:

**Step 1:** The gradient flows from the loss all the way back to the hidden state:
$$\frac{\partial L}{\partial \mathbf{h}} \quad \leftarrow \text{computed by the RNN's own backprop (BPTT)}$$

**Step 2:** The hidden state $\mathbf{h}$ depends on $\mathbf{e}_i$ through the RNN's input weight matrix $W_{ih}$ and $\tanh$:

$$\mathbf{h} = \tanh(W_{ih} \cdot \mathbf{e}_i + W_{hh} \cdot \mathbf{h}_{t-1} + b)$$

So:
$$\frac{\partial \mathbf{h}}{\partial \mathbf{e}_i} = (1 - \mathbf{h}^2) \cdot W_{ih}$$

(derivative of $\tanh$ is $(1 - \tanh^2)$, and $W_{ih}$ comes from the linear transform of the embedding)

**Full equation:**

$$\boxed{\frac{\partial L}{\partial \mathbf{e}_i} = \frac{\partial L}{\partial \mathbf{h}} \cdot (1 - \mathbf{h}^2) \cdot W_{ih}}$$

---

### The Key Insight: Sparse Gradients

The update rule for the embedding matrix $E$ is:

$$E[i] \leftarrow E[i] - \eta \cdot \frac{\partial L}{\partial \mathbf{e}_i}$$

But notice what happens for **all other rows $j \neq i$** (words not used in this batch):

$$\frac{\partial L}{\partial \mathbf{e}_j} = \mathbf{0} \quad \text{(zero vector)}$$

Because the loss has **zero dependency** on an embedding that was never looked up. This is what makes the gradient **sparse** — only the rows corresponding to words that appeared in the current batch receive any update. This is why training is efficient even with a vocabulary of 100,000 words!

---

### Summary Table

| Symbol | Meaning |
|:---|:---|
| $E[i]$ | Embedding vector for word $i$ (what gets updated) |
| $\frac{\partial L}{\partial \mathbf{h}}$ | Gradient arriving from the RNN (via BPTT) |
| $(1 - \mathbf{h}^2)$ | Derivative of $\tanh$ at the hidden state |
| $W_{ih}$ | RNN's input-to-hidden weight matrix |
| $\eta$ | Learning rate |

> [!NOTE]
> In practice, PyTorch never explicitly computes this formula manually. Instead, the autograd engine traces the computation graph and applies this chain rule automatically when you call `loss.backward()`.